# Lab 3 — one controlled change and a defensible recommendation

**Authors:** Matías Mouat and María José López

100 points · 6% NP · same pairs as Lab 2 · due Thu 29 October 2026, 12:30 Santiago.

Lab 1 checked data and simple rules. Lab 2 changed one input under fixed comparison conditions.
Here you keep the inputs fixed and change one model setting, then explain errors and write for a
Head of Strategy. The supplied code runs the experiment; you must supply the checked explanation,
one-change specification, three supported failure patterns, memo and truthful AI record.

Follow the numbered headings. See README for setup and GLOSSARY for terms.
A losing additional candidate can earn full credit. The unchanged starter is incomplete.


## Environment and data check

Extract the whole ZIP and choose your kernel before running this block. Follow README.md if a check fails.

In [1]:
from pathlib import Path
import sys
import importlib
print('Selected Python:', sys.executable)
print('Version:', sys.version.split()[0], '| working directory:', Path.cwd())
required_files = ['lab03_support_v3.py', 'pitwindow_student_v2.py', 'data/learning_v2.csv']
missing = [name for name in required_files if not Path(name).is_file()]
if missing:
    raise FileNotFoundError('Extract the complete ZIP and open its folder as cwd. Missing: ' + ', '.join(missing))
for module in ['numpy', 'pandas', 'sklearn', 'scipy', 'joblib']:
    loaded = importlib.import_module(module)
    print(module, loaded.__version__)
print('Ready: local files/imports found. Next cell checks data integrity and split.')


Selected Python: D:\Universidad\Trimeste 3 2026\.venv\Scripts\python.exe
Version: 3.11.9 | working directory: D:\Universidad\Trimeste 3 2026
numpy 2.0.2


pandas 2.3.1


sklearn 1.8.0
scipy 1.17.1
joblib 1.4.2
Ready: local files/imports found. Next cell checks data integrity and split.


In [2]:
from pathlib import Path
import pandas as pd
from lab03_support_v3 import (
    FEATURES, SEED, BUDGET, load_learning, run_base_comparison,
    run_one_change, export_run, markdown_table,
)
ROOT = Path('.')
train, dev = load_learning(ROOT)
print('Seed:', SEED, '| development alert budget:', BUDGET)
print('Permitted inputs:', FEATURES)
audit = pd.DataFrame([
    {'split': name, 'rows': len(df), 'positive_windows': int(df.y.sum()),
     'eligible_events': df.loc[df.y.eq(1), 'event_id'].nunique(),
     'first_year': int(df.year.min()), 'last_year': int(df.year.max())}
    for name, df in [('train', train), ('dev', dev)]
])
display(audit)
assert set(train.opportunity_id).isdisjoint(dev.opportunity_id)


Seed: 414 | development alert budget: 0.1
Permitted inputs: ['circuit_id', 'compound', 'tyre_life', 'lap_number', 'progress', 'prior_pit_entries', 'pace_median_s', 'pace_slope_s_per_lap', 'pace_n']


,split,rows,positive_windows,eligible_events,first_year,last_year
0,train,63165,5523,1934,2018,2021
1,dev,9451,1073,370,2022,2022


## 1. Define the prediction

**Our answer:**

We predict at the end of a rival driver's lap L, using only what is recorded up to the cut time `cut_s` of that lap. A row is positive (`y = 1`) when that same driver enters the pits during their own next three laps (L+1 to L+3). It is an observed pit entry, not a statement that the stop was the right strategy call.

Allowed input example: `tyre_life`, the last reported age of the current tyre set at lap L. It is known at prediction time, although it can lag a little because it is the last reported value.

Future-known field kept out of X: `event_id` (and `event_s`, the time of the actual pit entry). It only exists because the stop happened later, so using it would leak the answer. We only use it to count distinct events and audit labels. `future_neutralisation` is also future information and is not in `FEATURES`.

What the snapshot can and cannot show: the dataset is a corrected historical rebuild from public timing. It can tell us whether the inputs are *logically* available at the end of lap L. It cannot tell us when each value would really arrive in a live feed, or whether live data would have the same corrections, so any live delay is untested.

## Practice reading the counts

Illustrative manual example only, NOT F1 data. Two positive windows refer to event A; a third refers
to event B. The final window is negative and triggers a false alert.

| Window | y | Event | Alert |
|---|---:|---|---|
| w1 | 1 | A | yes |
| w2 | 1 | A | yes |
| w3 | 1 | B | no |
| w4 | 0 | none | yes |

Ungraded check: count TP, FP, FN, alerts, eligible/caught events, precision and event recall.
What changes if w2 stops alerting? Think first, then open the self-check below.


### Optional practice response

Before opening the self-check below, write or think through: TP = __; FP = __; FN = __; alerts = __; eligible/caught events = __ / __; precision = __; event recall = __. If w2 becomes quiet: __. This exercise is ungraded; no separate submission is required.

<details>
<summary>Open after trying: self-check for the four fictional windows</summary>

TP=2, FP=1, FN=1, TN=0; alerts=3; precision=2/3. There are two eligible events A/B, one caught, so event recall=1/2. If w2 becomes quiet: TP=1, FP=1, FN=2, alerts=2, precision=1/2; A is still caught by w1 and event recall remains 1/2.

</details>

In [3]:
base_comparison, fitted, base_policies = run_base_comparison(train, dev)
compact_columns = ['model', 'alerts', 'precision', 'events_caught', 'eligible_events', 'event_recall']
print('Start with DEVELOPMENT. Undefined precision means there were no alerts.')
display(base_comparison.loc[base_comparison.split.eq('dev'), compact_columns])
columns = ['model', 'split', 'n', 'TP', 'FP', 'FN', 'TN', 'alerts', 'alert_rate',
           'precision', 'events_caught', 'eligible_events', 'event_recall', 'threshold']
print("Audit view: train and development reuse the threshold selected for that model on dev.")
display(base_comparison[columns])


Start with DEVELOPMENT. Undefined precision means there were no alerts.


,model,alerts,precision,events_caught,eligible_events,event_recall
1,never_alert,0,NaN,0,370,0.000000
3,historical_baseline,931,0.268528,94,370,0.254054
5,tree,571,0.269702,63,370,0.170270
7,forest_all,938,0.276119,107,370,0.289189
9,forest_sqrt,945,0.359788,140,370,0.378378


Audit view: train and development reuse the threshold selected for that model on dev.


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
0,never_alert,train,63165,0,0,5523,57642,0,0.000000,NaN,0,1934,0.000000,NaN
1,never_alert,dev,9451,0,0,1073,8378,0,0.000000,NaN,0,370,0.000000,NaN
2,historical_baseline,train,63165,3042,6645,2481,50997,9687,0.153360,0.314029,1137,1934,0.587901,0.161926
3,historical_baseline,dev,9451,250,681,823,7697,931,0.098508,0.268528,94,370,0.254054,0.161926
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
6,forest_all,train,63165,2503,4503,3020,53139,7006,0.110916,0.357265,969,1934,0.501034,0.718176
7,forest_all,dev,9451,259,679,814,7699,938,0.099249,0.276119,107,370,0.289189,0.718176
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283


## 2. Explain the common comparison

**Our answer:**

On development (9,451 windows, 1,073 positive windows, 370 eligible pit entries), `forest_sqrt` caught 140 / 370 entries (event recall 0.378) with 945 alerts, of which 340 were correct (precision 340 / 945 = 0.360). `forest_all` caught 107 / 370 (0.289) with 938 alerts and precision 259 / 938 = 0.276. Both used almost the full budget of 945 alerts (10% of 9,451), so the gap is not from alerting more. The historical baseline caught 94 / 370 and the single tree only 63 / 370.

The only difference between the two forests is `max_features`. After preprocessing there are 48 columns (numeric inputs, missing-value indicators and one-hot columns for circuit and compound). `forest_all` looks at all 48 columns at every split, so its trees tend to pick the same strong columns first and end up looking alike. `forest_sqrt` looks at a random 6 of the 48 at each split. Every column is still available to the forest; it is just not offered at every split. This gives the trees more chances to use other inputs. We think this variety helps when averaging, but we did not measure how correlated the trees are, so that part is a hypothesis.

`never_alert` has precision 0 / 0, which is undefined, not zero: there are no alerts to be right or wrong about. TP windows are not the same as caught events. One pit entry can have up to three positive windows (2.9 on average in dev), so `forest_sqrt` has 340 TP windows but only 140 distinct caught entries. Every model uses the same rule (at most 10% alerts on dev, maximise caught entries), but the numeric threshold is chosen per model: 0.608 for `forest_sqrt`, 0.718 for `forest_all`. Scores are on different scales, so the same cutoff would not mean the same number of alerts. The tree only reached 571 alerts because many windows share the same tree score; adding the next tied group would break the 945 limit, so part of the budget stays unused.

Train/dev gap: `forest_sqrt` has event recall 0.520 on train (1,005 / 1,934) against 0.378 on dev. Train is in-sample, and the dev threshold reused on train produces 14.4% alerts there, more than the 10% on dev. So part of the gap comes from extra alerts on train, and part may be overfitting or real changes in 2022 (new cars, new tyre range). The gap alone does not tell us which one. Dev is also not a clean test, because we used it to pick the threshold.

## 3. Record your expectation before your additional run

The base outputs are already supplied. Complete `experiment_spec.md` before running YOUR candidate
for the first time. If already run, say so and keep a retrospective account. Do not invent history.

Choose ONE tuple: tree / max_depth / 6 (old 3); forest_sqrt / max_depth / 3 (old 6);
or forest_sqrt / min_samples_leaf / 100 (old 30). No search for the best of all three.
The helper clones the base pipeline and changes one setting. Its threshold is reselected on dev
under the same policy, so the numeric threshold may differ. This compares the resulting alert systems.


Choose one additional configuration. Repeating that same configuration for debugging or Restart + Run All is allowed. Disclose any different configurations you already tried; do not hide a search.

In [4]:
EXPERIMENT_MODEL_ID = 'forest_sqrt'  # 'tree' or 'forest_sqrt', matching the allowed table
EXPERIMENT_PARAMETER = 'min_samples_leaf'  # 'max_depth' or 'min_samples_leaf'
EXPERIMENT_VALUE = 100      # the one allowed new value


In [5]:
change = None
policies = dict(base_policies)
comparison = base_comparison.copy()
if any(value is None for value in [EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE]):
    print('INCOMPLETE STARTER: configure one additional experiment in extra-config.')
else:
    extra_rows, extra_model, extra_policy, change = run_one_change(
        train, dev, fitted, EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE)
    comparison = pd.concat([base_comparison, extra_rows], ignore_index=True)
    policies[change['model_id']] = extra_policy
    display(comparison.loc[comparison.model.isin([change['parent'], change['model_id']]), columns])


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283
10,forest_sqrt__min_samples_leaf_100,train,63165,2716,6900,2807,50742,9616,0.152236,0.282446,1010,1934,0.522234,0.601914
11,forest_sqrt__min_samples_leaf_100,dev,9451,376,562,697,7816,938,0.099249,0.400853,159,370,0.429730,0.601914


## 4. Explain your additional result

**Our answer:**

Candidate `forest_sqrt__min_samples_leaf_100` against its parent `forest_sqrt` on dev:

| | parent (leaf 30) | candidate (leaf 100) |
|---|---|---|
| Entries caught / eligible | 140 / 370 (0.378) | 159 / 370 (0.430) |
| Alerts (TP + FP) | 945 (340 + 605) | 938 (376 + 562) |
| Precision (TP / alerts) | 0.360 | 0.401 |
| Selected threshold | 0.608 | 0.602 |
| AP (supplementary) | 0.331 | 0.344 |

On train the candidate caught 1,010 / 1,934 (0.522) against 1,005 / 1,934 (0.520), with lower train precision (0.282 vs 0.294) and lower train AP (0.297 vs 0.310).

Did our expectation hold? Partly. We expected a small change (within about ±10 entries) and a slight drop on train. In fact dev improved by 19 entries with 7 fewer alerts, so the effect was bigger than we expected. Train event recall did not drop (+5 entries), but train precision and train AP did drop. That fits the idea of less in-sample fitting, but it is weaker than we predicted. The alert count stayed near the cap, as expected.

Possible mechanism: with leaves of at least 100 rows, each tree can no longer create small leaves for narrow 2018–2021 situations. Scores become averages over larger groups of laps, and that seems to rank 2022 windows a bit better. The AP change (0.331 → 0.344) does not depend on the threshold, so the improvement is not only from the new threshold. We did not inspect individual leaves, so "small leaves were memorising old seasons" is still a hypothesis. The threshold also changed (0.608 → 0.602), so the final alert results come from both the new model and its new threshold.

Bounded recommendation: the candidate is the better of the two on this dev period, and we would keep it over the parent for further checking. It is not ready for use yet. Dev is 11 races and was used to choose the threshold, so +19 entries (about 5 percentage points) may partly be luck in those races. The data is also a historical rebuild and says nothing about live timing. The next step is one check on held-out races that were not used here, run by whoever controls that data.

## 5. Explain three distinct failure patterns

Keep your additional candidate and its selected threshold frozen. Inspect the supplied tables, choose the denominator for your claim, and use the queries to find supporting cases. This step does not train or retune.

### Choose your denominator

| Question within your context | Calculation | Denominator counts |
|---|---|---|
| Of the alerts issued, how many were false? | FP / (TP + FP) | Alerted windows; false-alert fraction, equal to 1 − precision when defined. |
| Of the negative windows, how many alerted? | FP / (FP + TN) | Negative windows; false positive rate (FPR). |
| Of the positive windows, how many were quiet? | FN / (TP + FN) | Positive windows; `miss_rate` in the context export. |
| Of the eligible events, how many were completely missed? | Completely missed events / eligible events | Distinct pit entries; use all positive windows for each event. |

A zero denominator means undefined, shown as NaN; it does not mean zero errors. Name your population and unit with every fraction. Window context groups do not define an event population automatically: if describing missed events in a context, state which distinct events you include and check all their positive windows, including those outside that context. The event query below checks a whole event.


### A complete entry from the four fictional windows

This uses only the four-window practice example, not the Rival dataset or an additional candidate's results.

“For the context of all four toy windows, w4 is a false alert (y=0, alert=yes). One of the three alerts is false: FP/alerts=1/3. The traceable case is `opportunity_id=w4`. This could prompt an unnecessary review of a rival that does not enter within the horizon; next check the time and outcome around w4 before attributing a cause. With only four invented windows, this illustrates the reasoning format and cannot establish a general effect.”

The same counts give FPR=1/(1+0)=1/1 over negative windows, and miss_rate=1/(2+1)=1/3 over positive windows. Those fractions answer different questions. Event B has one positive window (w3), none alerted: one of two eligible events is completely missed. A quiet w2 alone would not make event A missed because w1 still alerts.


In [6]:
from lab03_support_v3 import development_predictions, context_error_summary, event_error_summary
predictions = None
if change is None:
    print('INCOMPLETE: run your additional candidate before examining its errors.')
else:
    predictions = development_predictions(dev, extra_model, extra_policy, change['model_id'])
    context_errors = context_error_summary(predictions)
    event_errors = event_error_summary(predictions)
    print('Descriptive context tables. No refit or threshold change occurs here.')
    display(context_errors)
    print('FP window examples (highest scores first):')
    display(predictions.loc[predictions.error_type.eq('FP')].sort_values('score', ascending=False).head(8))
    print('FN window examples (lowest scores first):')
    display(predictions.loc[predictions.error_type.eq('FN')].sort_values('score').head(8))
    print('Missed events:', int(event_errors.missed_event.sum()), '/', len(event_errors))
    display(event_errors.loc[event_errors.missed_event].head(8))


Descriptive context tables. No refit or threshold change occurs here.


,dimension,context,n,positive_windows,alerts,TP,FP,FN,TN,precision,miss_rate
0,compound,HARD,4037,266,274,44,230,222,3541,0.160584,0.834586
1,compound,INTERMEDIATE,377,86,62,36,26,50,265,0.580645,0.581395
2,compound,MEDIUM,3593,473,421,189,232,284,2888,0.448931,0.600423
3,compound,SOFT,1211,201,138,82,56,119,954,0.594203,0.592040
4,compound,UNKNOWN,4,0,0,0,0,0,4,NaN,NaN
5,compound,WET,229,47,43,25,18,22,164,0.581395,0.468085
6,circuit_id,10,797,61,104,42,62,19,674,0.403846,0.311475
7,circuit_id,144,720,88,73,30,43,58,589,0.410959,0.659091
8,circuit_id,149,617,55,87,33,54,22,508,0.379310,0.400000
9,circuit_id,15,1032,159,35,19,16,140,857,0.542857,0.880503


FP window examples (highest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
3623,2022-05-08:3:23:5956.816000,2022-05-08,3,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.729027,True,FP
3626,2022-05-08:3:26:6242.013000,2022-05-08,3,26,151,MEDIUM,25.0,0.456140,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.715471,True,FP
557,2022-03-20:4:17:5503.620000,2022-03-20,4,17,63,MEDIUM,16.0,0.298246,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.710303,True,FP
560,2022-03-20:4:20:5810.495000,2022-03-20,4,20,63,MEDIUM,19.0,0.350877,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.708626,True,FP
3305,2022-05-08:11:20:5636.771000,2022-05-08,11,20,151,MEDIUM,19.0,0.350877,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.705372,True,FP
3625,2022-05-08:3:25:6147.017000,2022-05-08,3,25,151,MEDIUM,24.0,0.438596,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.702046,True,FP
4012,2022-05-08:77:21:5739.305000,2022-05-08,77,21,151,MEDIUM,20.0,0.368421,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.701659,True,FP
3880,2022-05-08:55:23:5911.352000,2022-05-08,55,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.701486,True,FP


FN window examples (lowest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
2455,2022-04-24:16:51:8308.778000,2022-04-24,16,51,6,SOFT,1.0,0.809524,2022-04-24:16:pit:8477.893000,1.0,forest_sqrt__min_samples_leaf_100,0.168912,False,FN
8576,2022-07-10:16:56:7713.940000,2022-07-10,16,56,19,HARD,6.0,0.788732,2022-07-10:16:pit:7863.977000,1.0,forest_sqrt__min_samples_leaf_100,0.172992,False,FN
8517,2022-07-10:14:55:7702.648000,2022-07-10,14,55,19,HARD,0.0,0.774648,2022-07-10:14:pit:7850.818000,1.0,forest_sqrt__min_samples_leaf_100,0.175046,False,FN
8518,2022-07-10:14:56:7773.231000,2022-07-10,14,56,19,HARD,0.0,0.788732,2022-07-10:14:pit:7850.818000,1.0,forest_sqrt__min_samples_leaf_100,0.175046,False,FN
8577,2022-07-10:16:57:7782.783000,2022-07-10,16,57,19,HARD,7.0,0.802817,2022-07-10:16:pit:7863.977000,1.0,forest_sqrt__min_samples_leaf_100,0.175175,False,FN
8374,2022-07-10:1:55:7648.611000,2022-07-10,1,55,19,HARD,1.0,0.774648,2022-07-10:1:pit:7869.472000,1.0,forest_sqrt__min_samples_leaf_100,0.176645,False,FN
8376,2022-07-10:1:57:7788.187000,2022-07-10,1,57,19,HARD,1.0,0.802817,2022-07-10:1:pit:7869.472000,1.0,forest_sqrt__min_samples_leaf_100,0.179797,False,FN
8375,2022-07-10:1:56:7718.344000,2022-07-10,1,56,19,HARD,1.0,0.788732,2022-07-10:1:pit:7869.472000,1.0,forest_sqrt__min_samples_leaf_100,0.179797,False,FN


Missed events: 211 / 370


,event_id,race_id,driver_id,positive_windows,alerted_positive_windows,caught,missed_event
4,2022-03-20:11:pit:8037.416000,2022-03-20,11,3,0,False,True
6,2022-03-20:14:pit:6293.550000,2022-03-20,14,3,0,False,True
7,2022-03-20:14:pit:8006.856000,2022-03-20,14,3,0,False,True
10,2022-03-20:16:pit:8333.486000,2022-03-20,16,3,0,False,True
13,2022-03-20:18:pit:8372.487000,2022-03-20,18,2,0,False,True
16,2022-03-20:1:pit:8020.852000,2022-03-20,1,3,0,False,True
17,2022-03-20:1:pit:9348.693000,2022-03-20,1,3,0,False,True
20,2022-03-20:20:pit:8430.179000,2022-03-20,20,2,0,False,True


### Find the cases behind your claim

Set CONTEXT_COLUMN to a supplied context column (for example compound or circuit_id), CONTEXT_VALUE to an actual value of the same type shown in your predictions, and ERROR_KIND to FP or FN. The summary uses all rows in that context, so its denominators do not accidentally count only errors. `cases` contains every matching case; the display previews eight. For three patterns you can repeat the query with recorded settings, or copy this query block within this section. Keep each setting and relevant output alongside its written claim.

For a whole-event claim, copy an event_id from the missed-event table into EVENT_TO_INSPECT. The second query retrieves all its positive windows, including their opportunity_id values, without a context filter. An event is missed only if none alerts. Unknown IDs return no rows and are not evidence of a missed event.


In [7]:
from lab03_support_v3 import context_query, positive_windows_for_event

# Pattern 1: false alerts on HARD tyres
CONTEXT_COLUMN = 'compound'
CONTEXT_VALUE = 'HARD'
ERROR_KIND = 'FP'

if predictions is None:
    print('Run your additional candidate first; its frozen predictions are needed.')
elif CONTEXT_COLUMN is None or CONTEXT_VALUE is None:
    print('Set the context column/value above to inspect your own error evidence.')
else:
    context_summary, cases = context_query(predictions, CONTEXT_COLUMN, CONTEXT_VALUE, ERROR_KIND)
    display(context_summary)
    print('Matching cases:', len(cases), '(first eight shown)')
    display(cases.sort_values('score', ascending=False).head(8))
    print('Tyre life of HARD alerts, median by outcome:')
    hard_alerts = predictions.loc[predictions.compound.eq('HARD') & predictions.alert]
    display(hard_alerts.groupby('error_type').tyre_life.describe())
    all_alerts = predictions.alert.sum()
    print('All dev alerts false:', int(predictions.error_type.eq('FP').sum()), '/', int(all_alerts))

,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,compound,HARD,4037,44,230,222,3541,274,3771,266,0.839416,0.060992,0.834586


Matching cases: 230 (first eight shown)


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
3846,2022-05-08:5:35:7111.834000,2022-05-08,5,35,151,HARD,36.0,0.614035,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.678954,True,FP
3842,2022-05-08:5:31:6729.428000,2022-05-08,5,31,151,HARD,32.0,0.543860,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.677202,True,FP
3843,2022-05-08:5:32:6825.152000,2022-05-08,5,32,151,HARD,33.0,0.561404,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.676659,True,FP
3448,2022-05-08:18:30:6630.781000,2022-05-08,18,30,151,HARD,31.0,0.526316,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.675898,True,FP
3979,2022-05-08:63:34:6983.540000,2022-05-08,63,34,151,HARD,33.0,0.596491,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.674404,True,FP
1181,2022-03-27:27:25:6423.894000,2022-03-27,27,25,149,HARD,25.0,0.500000,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.672308,True,FP
3839,2022-05-08:5:28:6442.307000,2022-05-08,5,28,151,HARD,29.0,0.491228,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.672256,True,FP
3972,2022-05-08:63:27:6325.321000,2022-05-08,63,27,151,HARD,26.0,0.473684,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.671762,True,FP


Tyre life of HARD alerts, median by outcome:


,count,mean,std,min,25%,50%,75%,max
error_type,,,,,,,,
FP,230.0,29.160870,5.330577,20.0,25.0,28.5,33.0,44.0
TP,44.0,35.204545,4.668447,27.0,31.0,36.0,38.0,45.0


All dev alerts false: 562 / 938


In [8]:
# Pattern 2 (copy of the query block): missed windows at circuit 19 (race 2022-07-10)
CONTEXT_COLUMN_2 = 'circuit_id'
CONTEXT_VALUE_2 = '19'
ERROR_KIND_2 = 'FN'

if predictions is not None:
    context_summary_2, cases_2 = context_query(predictions, CONTEXT_COLUMN_2, CONTEXT_VALUE_2, ERROR_KIND_2)
    display(context_summary_2)
    print('Matching cases:', len(cases_2), '(highest scores first: near misses)')
    display(cases_2.sort_values('score', ascending=False).head(8))
    print('Median score by label, circuit 19 vs all dev:')
    display(pd.DataFrame({
        'circuit_19': predictions.loc[predictions.circuit_id.eq('19')].groupby('y').score.median(),
        'all_dev': predictions.groupby('y').score.median()}))
    events_19 = event_errors.loc[event_errors.race_id.eq('2022-07-10')]
    print('Circuit 19 entries completely missed:', int(events_19.missed_event.sum()), '/', len(events_19))

,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,circuit_id,19,1126,12,14,114,986,26,1000,126,0.538462,0.014,0.904762


Matching cases: 114 (highest scores first: near misses)


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
8728,2022-07-10:22:24:5477.055000,2022-07-10,22,24,19,HARD,23.0,0.338028,2022-07-10:22:pit:5622.461000,1.0,forest_sqrt__min_samples_leaf_100,0.586787,False,FN
8488,2022-07-10:14:24:5479.942000,2022-07-10,14,24,19,HARD,23.0,0.338028,2022-07-10:14:pit:5696.507000,1.0,forest_sqrt__min_samples_leaf_100,0.586449,False,FN
8727,2022-07-10:22:23:5404.606000,2022-07-10,22,23,19,HARD,22.0,0.323944,2022-07-10:22:pit:5622.461000,1.0,forest_sqrt__min_samples_leaf_100,0.575868,False,FN
8846,2022-07-10:24:23:5405.819000,2022-07-10,24,23,19,HARD,22.0,0.323944,2022-07-10:24:pit:5481.077000,1.0,forest_sqrt__min_samples_leaf_100,0.575868,False,FN
8957,2022-07-10:31:15:4812.772000,2022-07-10,31,15,19,MEDIUM,14.0,0.211268,2022-07-10:31:pit:4884.900000,1.0,forest_sqrt__min_samples_leaf_100,0.568373,False,FN
8490,2022-07-10:14:26:5623.975000,2022-07-10,14,26,19,HARD,25.0,0.366197,2022-07-10:14:pit:5696.507000,1.0,forest_sqrt__min_samples_leaf_100,0.565852,False,FN
8489,2022-07-10:14:25:5552.388000,2022-07-10,14,25,19,HARD,24.0,0.352113,2022-07-10:14:pit:5696.507000,1.0,forest_sqrt__min_samples_leaf_100,0.560235,False,FN
8845,2022-07-10:24:22:5332.618000,2022-07-10,24,22,19,HARD,21.0,0.309859,2022-07-10:24:pit:5481.077000,1.0,forest_sqrt__min_samples_leaf_100,0.559940,False,FN


Median score by label, circuit 19 vs all dev:


,circuit_19,all_dev
y,,
0.0,0.259936,0.374560
1.0,0.290521,0.565343


Circuit 19 entries completely missed: 38 / 42


In [9]:
# Pattern 3: whole-event check for entries made on fresh tyres (descriptive only, no refit)
if predictions is not None:
    positives = predictions.loc[predictions.y.eq(1)]
    per_event = positives.groupby('event_id').agg(
        positive_windows=('opportunity_id', 'size'), max_tyre_life=('tyre_life', 'max'),
        max_score=('score', 'max'), caught=('alert', 'any'), race_id=('race_id', 'first'))
    fresh = per_event.loc[per_event.max_tyre_life.le(5)]
    print('Entries whose positive windows ALL have tyre_life <= 5:', len(fresh), '/', len(per_event))
    print('Of these, completely missed:', int((~fresh.caught).sum()), '/', len(fresh))
    print('Highest score among their windows:', round(fresh.max_score.max(), 3),
          '| selected threshold:', round(extra_policy['threshold'], 3))
    print('Overlap with circuit 19 (race 2022-07-10):', int(fresh.race_id.eq('2022-07-10').sum()), 'entries')
    display(fresh.sort_values('max_score', ascending=False).head(8))

Entries whose positive windows ALL have tyre_life <= 5: 49 / 370
Of these, completely missed: 49 / 49
Highest score among their windows: 0.524 | selected threshold: 0.602
Overlap with circuit 19 (race 2022-07-10): 21 entries


,positive_windows,max_tyre_life,max_score,caught,race_id
event_id,,,,,
2022-06-19:20:pit:4293.490000,3,5.0,0.524388,False,2022-06-19
2022-05-29:5:pit:8158.727000,3,4.0,0.504415,False,2022-05-29
2022-05-29:22:pit:8161.347000,3,4.0,0.504415,False,2022-05-29
2022-05-08:24:pit:4346.853000,3,4.0,0.503206,False,2022-05-08
2022-06-19:10:pit:4144.600000,3,3.0,0.469140,False,2022-06-19
2022-06-19:5:pit:4142.435000,3,4.0,0.469028,False,2022-06-19
2022-07-03:3:pit:10604.515000,3,4.0,0.407932,False,2022-07-03
2022-07-10:77:pit:4178.848000,3,4.0,0.406652,False,2022-07-10


In [10]:
# Whole-event check for one circuit-19 near miss used in pattern 2
EVENT_TO_INSPECT = '2022-07-10:14:pit:5696.507000'

if predictions is None:
    print('Run your additional candidate first.')
elif EVENT_TO_INSPECT is None:
    print('Set an event_id above when checking a completely missed event.')
else:
    event_windows = positive_windows_for_event(predictions, EVENT_TO_INSPECT)
    display(event_windows)
    if event_windows.empty:
        print('No positive windows found: check the ID; do not call this a missed event.')
    else:
        print('Positive windows:', len(event_windows), 'Alerted:', int(event_windows.alert.sum()))
        print('Completely missed event:', not bool(event_windows.alert.any()))

# Same check for one fresh-tyre entry used in pattern 3
fresh_windows = positive_windows_for_event(predictions, '2022-04-10:18:pit:4036.326000')
display(fresh_windows)
print('Fresh-tyre example completely missed:', not bool(fresh_windows.alert.any()))

,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
8488,2022-07-10:14:24:5479.942000,2022-07-10,14,24,19,HARD,23.0,0.338028,2022-07-10:14:pit:5696.507000,1.0,forest_sqrt__min_samples_leaf_100,0.586449,False,FN
8489,2022-07-10:14:25:5552.388000,2022-07-10,14,25,19,HARD,24.0,0.352113,2022-07-10:14:pit:5696.507000,1.0,forest_sqrt__min_samples_leaf_100,0.560235,False,FN
8490,2022-07-10:14:26:5623.975000,2022-07-10,14,26,19,HARD,25.0,0.366197,2022-07-10:14:pit:5696.507000,1.0,forest_sqrt__min_samples_leaf_100,0.565852,False,FN


Positive windows: 3 Alerted: 0
Completely missed event: True


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
1659,2022-04-10:18:2:3923.790000,2022-04-10,18,2,10,HARD,2.0,0.034483,2022-04-10:18:pit:4036.326000,1.0,forest_sqrt__min_samples_leaf_100,0.274748,False,FN


Fresh-tyre example completely missed: True


### What counts as a separate pattern

The same error type in different contexts can count as separate patterns when each entry states its context rule, gives its own count/denominator and traceable case, and explains a consequence and next check. Three cases of the same error in the same context count as one pattern. Changing only a context name without evidence and interpretation is incomplete. You do not need three different causes, independent groups or statistical significance. Declare overlap; do not add overlapping counts as distinct errors.

Fictional examples for interpreting the requirement (these are not Rival results):

- Acceptable entry A: “For context `compound == TOY_A`, 2 of 6 alerts are false (FP/alerts), including `toy-A-4`. These alerts require checking a rival that does not enter within the horizon; next check the neighbouring windows to see whether alerts repeat around the same situation. The six-alert sample is small.”
- Acceptable entry B: “For context `compound == TOY_B`, 3 of 10 alerts are false, including `toy-B-2`. The review burden also occurs here; next inspect the recorded timing around those cases before proposing a cause.” With their separate supporting rows, A and B count as two patterns even though both are FP. Your submission must provide the actual rows behind its own claims.
- Incomplete: “TOY_A has false alerts; TOY_B has false alerts; TOY_C has false alerts.” Context names alone supply neither counts/denominators nor cases, consequences or checks. Three IDs from TOY_A under the same rule also describe one pattern.


### Your three entries

All counts come from the frozen candidate `forest_sqrt__min_samples_leaf_100` on dev (threshold 0.602) and the query cells above. Nothing was refit or retuned.

1. **False alerts on HARD tyres (FP, `compound == HARD`).** 230 of 274 HARD alerts were false (FP / alerts = 0.84), compared with 562 / 938 = 0.60 over all dev alerts. Example: `2022-05-08:5:35:7111.834000` (driver 5, lap 35, tyre life 36, score 0.679, no stop in the next three laps). Every HARD false alert had tyre life of 20 laps or more (median 28.5). Observed: the model alerts on old HARD tyres and most of those rivals keep going. Hypothesis, not measured: it learned "old tyre means stop soon", but a hard set is often meant to last to the end. Consequence: the strategist would mostly chase false HARD warnings and might start ignoring them. Next check: compare tyre life of HARD stints that ended in a stop vs ones that ran to the flag in the 2018–2021 train data.

2. **Missed windows at circuit 19 (FN, `circuit_id == '19'`, race 2022-07-10).** 114 of 126 positive windows were missed (miss rate 0.90, against 697 / 1,073 = 0.65 overall). At event level, 38 of 42 entries were completely missed. Only 26 alerts went off in the whole race. Example: `2022-07-10:14:24:5479.942000` (HARD, tyre life 23, score 0.586, just under 0.602). The event check shows none of entry `2022-07-10:14:pit:5696.507000`'s three windows alerted. Observed: at this circuit the median score of positive windows (0.29) is almost the same as negatives (0.26), so the model barely separates them there. Overlap: 21 of the 42 entries also belong to pattern 3; leaving those out, 17 of 21 are still missed. Hypothesis: this race's stop timing differed from the 2018–2021 history for that circuit. Consequence: for a whole race the alert would be almost silent, and the team would get no warning. Next check: compare stop laps at this circuit in train vs 2022 before trusting the alert there.

3. **Completely missed entries on fresh tyres (missed events, all positive windows with `tyre_life <= 5`).** 49 of 370 eligible entries fall in this group and all 49 were completely missed (49 / 49). Their highest score was 0.524, under the 0.602 threshold. Example: entry `2022-04-10:18:pit:4036.326000`, whose only positive window `2022-04-10:18:2:3923.790000` (lap 2, HARD, tyre life 2, score 0.275) did not alert. Observed: no stop made on tyres 5 laps old or less is flagged. This group includes very early stops (lap 2, as here) and stops soon after an earlier stop. Hypothesis: these are stops caused by things the inputs do not include (damage, safety car, weather change), so the model has no signal. Consequence: unplanned early stops are exactly the ones that can surprise the team, and the alert gives no help. Overlap: 21 of these 49 are at circuit 19 (pattern 2), so we do not add the counts. Next check: look at the race notes for some of these entries to see how many line up with a safety car or damage. That still uses only past data and does not retune anything.

Limits: each pattern comes from one 11-race dev period. Circuit 19 is one race, so it is a single case, not a general effect.

In [11]:
manifest = export_run(comparison, policies, change, ROOT, predictions=predictions)
print('Additional experiment executed:', manifest['experiment_complete'])
print('This flag does NOT check your reasoning, three patterns, memo or AI record.')
print('Systems:', comparison.model.nunique(), '| train/dev rows:', len(comparison))
print('Generated comparison_table.md and all five outputs/ files. No manual transcription needed.')


Additional experiment executed:

 True
This flag does NOT check your reasoning, three patterns, memo or AI record.
Systems: 6 | train/dev rows: 12
Generated comparison_table.md and all five outputs/ files. No manual transcription needed.


## 6. Memo and submission

Complete memo.md and PROMPTS.md. The comparison table is generated automatically.
Check CHECKLIST.md, restart + run all, save visible outputs and push the final commit.
Completed comparison: six systems / twelve train-dev rows, plus your additional candidate's diagnostics.
Canvas text entry: both names, accessible repository URL, final commit and notebook path.
The execution flag does not certify your explanation or completed documents.


## Use your feedback next time

We did not have a specific Lab 2 comment that fit this lab, so we picked a weakness ourselves that is easy to fall into in this lab: giving a rate without saying what it is out of. Here we tried to write the count and the denominator for every fraction, and to say whether it counts windows or entries. For the capstone we will write the expectation and commit it before the first run, as we did here.